# Video Face Swap Demo - 2 người (inswapper + InsightFace)

**Logic:** 1 video mẫu có 2 người (ví dụ cảnh hôn nhau) + 2 ảnh khuôn mặt (người A, người B) → video mới giữ nguyên chuyển động/nền gốc, mỗi người trong video được thay đúng bằng khuôn mặt tương ứng đã cung cấp.

**Công nghệ dùng:**
- `insightface` (buffalo_l) để detect + align khuôn mặt từng frame
- `inswapper_128.onnx` để swap khuôn mặt
- `GFPGAN` (tùy chọn) để làm nét/phục hồi mặt sau khi swap
- `ffmpeg` để tách/ghép audio và dựng lại video

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

⚠️ Lưu ý: công nghệ face-swap có thể bị dùng sai mục đích (deepfake giả mạo người khác mà không có sự đồng ý). Chỉ dùng với ảnh/video của chính bạn hoặc người đã đồng ý, và cân nhắc gắn watermark/disclosure khi xuất bản sản phẩm thật.

## 0. ~~Cài môi trường Python 3.10 qua Miniconda~~ — HIỆN KHÔNG DÙNG ĐƯỢC

⚠️ **Cập nhật:** cách này hiện KHÔNG chạy được, vì bản thân package `condacolab` tự kiểm tra và chỉ chấp nhận Colab đang chạy đúng Python 3.12 (`assert colab_python == '3.12'` viết cứng trong code), trong khi Colab hiện đã lên Python 3.13 — `condacolab` chưa kịp cập nhật theo. Đây là giới hạn của chính tool `condacolab`, không phải do cấu hình sai.

**→ Bỏ qua toàn bộ mục 0, chuyển thẳng xuống mục 1.** Các cell ở mục 1 trở đi đã có sẵn patch để chạy được trên Python 3.13 của Colab hiện tại.

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

## 1. Cài đặt thư viện

In [ ]:
import sys
print('Python đang chạy trong cell này:', sys.version)

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.
# (insightface cũng khai báo opencv-python là dependency nên chắc chắn có cv2.)
!pip install -q onnxruntime-gpu gfpgan facexlib
!apt-get -qq install -y ffmpeg > /dev/null

### Fix riêng cho `basicsr` (bug với Python bản mới trên Colab)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy (thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json

os.makedirs('/tmp/basicsr_src', exist_ok=True)

# Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
# setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
    pkg_info = json.load(resp)

sdist_url = None
for url_info in pkg_info['urls']:
    if url_info['packagetype'] == 'sdist':
        sdist_url = url_info['url']
        break
assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

tar_name = sdist_url.split('/')[-1]
tar_path = f'/tmp/basicsr_src/{tar_name}'
urllib.request.urlretrieve(sdist_url, tar_path)
print(f'Đã tải: {tar_name}')

extract_dir = '/tmp/basicsr_build'
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(tar_path) as tar:
    # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace('.tar.gz', ''),
    # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
    root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
    assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
    root_name = root_names.pop()
    # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
    tar.extractall(extract_dir, filter='data')

pkg_dir = os.path.join(extract_dir, root_name)
setup_py_path = os.path.join(pkg_dir, 'setup.py')
assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

with open(setup_py_path, 'r') as f:
    content = f.read()

# Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
content = content.replace(
    "exec(compile(f.read(), version_file, 'exec'))",
    "exec(compile(f.read(), version_file, 'exec'), globals())"
)
content = content.replace(
    "return locals()['__version__']",
    "return globals()['__version__']"
)

with open(setup_py_path, 'w') as f:
    f.write(content)

print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
# sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
# thay vì lệnh `pip` bất kỳ đứng đầu PATH.
install_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
    capture_output=True, text=True
)
print(install_result.stdout[-3000:])
print(install_result.stderr[-3000:])
if install_result.returncode != 0:
    raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
print('Cài basicsr thành công.')

## 2. Tải model (inswapper + buffalo_l + GFPGAN)

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính sách, 
nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng (huggingface). 
Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ công rồi upload vào `/content/`.

In [ ]:
import os
os.makedirs('/content/models', exist_ok=True)

INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
GFPGAN_PATH = '/content/models/GFPGANv1.4.pth'

# Mirror cộng đồng trên Hugging Face (kiểm tra lại link còn sống trước khi chạy)
!wget -q -O {INSWAPPER_PATH} https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx

# GFPGAN weights cho face restoration (tùy chọn)
!wget -q -O {GFPGAN_PATH} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth

# wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra dung lượng.
# Nếu không, mãi tới cell load model mới nổ với lỗi onnx/torch rất khó đoán nguyên nhân.
def check_download(path, min_mb, hint):
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK  {os.path.basename(path)}: {size_mb:.1f} MB')

check_download(INSWAPPER_PATH, 200,
               'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')
check_download(GFPGAN_PATH, 300,
               'Kiểm tra lại link GitHub release của GFPGAN.')

!ls -lh /content/models/

## 3. Upload 2 ảnh khuôn mặt (người A, người B) + video mẫu (2 người)

In [ ]:
from google.colab import files

def pick_one(uploaded, what):
    names = list(uploaded.keys())
    assert len(names) > 0, f'Chưa upload {what} (bấm Cancel?). Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]

print('>> Upload ảnh khuôn mặt NGƯỜI A (rõ mặt, chính diện càng tốt):')
source_face_path_a = pick_one(files.upload(), 'ảnh mặt A')

print('\n>> Upload ảnh khuôn mặt NGƯỜI B:')
source_face_path_b = pick_one(files.upload(), 'ảnh mặt B')

print('\n>> Upload video mẫu (có 2 người, ví dụ cảnh hôn nhau):')
source_video_path = pick_one(files.upload(), 'video mẫu')

print(f'Ảnh mặt A: {source_face_path_a}')
print(f'Ảnh mặt B: {source_face_path_b}')
print(f'Video mẫu: {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

**Bước quan trọng:** cần xác định trong video, ai đứng bên trái / bên phải (theo frame đầu tiên có đủ 2 mặt) để biết gán ảnh A/B vào đúng người. Mặc định: **người A = mặt bên trái khung hình ở frame đầu tiên, người B = mặt bên phải**. Nếu bị ngược, đổi lại 2 ảnh upload ở bước 3 hoặc đảo `source_face_a`/`source_face_b` ở dưới.

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại `onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13), sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn, không dùng được GPU cho bước detect/swap qua ONNX).

In [ ]:
import subprocess, sys, importlib

def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode

if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu dùng CHUNG namespace `cv2`-style: cùng package `onnxruntime`.
    # Cài cái này đè cái kia là trạng thái hỏng đã biết (mất CUDAExecutionProvider, import lỗi loạn)
    # -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Dùng providers:', providers, '| ctx_id =', ctx_id)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

swapper = insightface.model_zoo.get_model('/content/models/inswapper_128.onnx',
                                          download=False, providers=providers)

def load_source_face(path, label):
    # cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
    # Phải chặn ngay, không thì app.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
    img = cv2.imread(path)
    assert img is not None, (
        f'Không đọc được ảnh {label} ({path}). Lưu lại thành .jpg/.png rồi upload lại.'
    )
    faces = app.get(img)
    assert len(faces) > 0, f'Không tìm thấy khuôn mặt trong ảnh {label}, thử ảnh khác rõ mặt hơn.'
    if len(faces) > 1:
        # Ảnh nguồn có nhiều mặt -> lấy mặt to nhất, vì thứ tự app.get() trả về không xác định.
        faces = sorted(faces,
                       key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]),
                       reverse=True)
        print(f'  (ảnh {label} có {len(faces)} mặt, dùng mặt lớn nhất)')
    return faces[0]

source_face_a = load_source_face(source_face_path_a, 'A')
source_face_b = load_source_face(source_face_path_b, 'B')

print('Đã detect khuôn mặt nguồn A và B thành công.')

### Fix riêng cho `basicsr` (bug với `torchvision` mới trên Colab)

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi `basicsr` (dependency của GFPGAN) vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi file `.py` còn tham chiếu module cũ (trong cả `basicsr`, `facexlib`, `gfpgan`).

Cell cũng tự **xoá các module hỏng khỏi `sys.modules`** cả trước lẫn sau khi vá — trước là để `find_spec` tra lại từ đĩa (một lần import hỏng trước đó có thể để lại `sys.modules['basicsr']` với `__spec__ = None`, làm `find_spec` ném lỗi), sau là để lần import kiểm chứng đọc đúng file vừa sửa. Nhờ vậy **không cần Runtime > Restart session** nữa: cell import GFPGAN bên dưới chạy được ngay.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    Xoá xong thì find_spec tra lại từ đĩa, và cell import bên dưới cũng chạy
    được ngay mà không cần Runtime > Restart session.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


purge_modules()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó.

    find_spec() chỉ định vị package, không chạy __init__.py -> không dính đúng cái
    ModuleNotFoundError mà ta đang muốn vá. (Bản cũ dùng `find` trong %%bash: glob
    không khớp thư mục nào thì find trả 1, pipefail + set -e giết cell trước khi in gì.)
    """
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


targets = {}
for name in PKGS:
    d = package_dir(name)
    if d is None:
        print(f'{name:9s}: CHƯA CÀI')
    else:
        print(f'{name:9s}: {d}')
        targets[name] = d

assert 'basicsr' in targets, (
    'Không tìm thấy basicsr. Chạy lại cell cài basicsr ở mục 1 rồi chạy lại cell này.'
)

patched = []
for name, d in targets.items():
    for p in d.rglob('*.py'):
        try:
            text = p.read_text(encoding='utf-8')
        except (UnicodeDecodeError, OSError):
            continue
        if OLD_MOD not in text:
            continue
        p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
        patched.append(p)

print()
if patched:
    for p in patched:
        print(f'đã vá: {p}')
else:
    print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn functional_tensor).')

# Purge lần nữa sau khi vá, để cell import bên dưới đọc lại file mới trên đĩa.
purge_modules()

# Kiểm chứng ngay tại đây thay vì để tới cell import gfpgan mới biết.
try:
    import basicsr.data.degradations
    print('\nOK: import basicsr.data.degradations thành công.')
except Exception as e:
    print(f'\nVẪN LỖI: {type(e).__name__}: {e}')
    print('Nếu lỗi vẫn liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại cell này.')
    raise

In [ ]:
import subprocess, sys, importlib

def try_import_gfpgan():
    importlib.invalidate_caches()
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception: gfpgan hỏng vì basicsr/torchvision thường ném ModuleNotFoundError,
        # nhưng tuỳ phiên bản torch cũng có thể là AttributeError/OSError.
        print(f'Chưa import được gfpgan: {type(e).__name__}: {e}')
        return False

GFPGAN_AVAILABLE = try_import_gfpgan()

if not GFPGAN_AVAILABLE:
    print('Thử cài lại gfpgan với log đầy đủ...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'gfpgan'],
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    GFPGAN_AVAILABLE = try_import_gfpgan()

if not GFPGAN_AVAILABLE:
    print('gfpgan không cài được. Sẽ BỎ QUA bước phục hồi mặt (USE_GFPGAN tự tắt ở cell dưới).')
    print('Pipeline chính (swap mặt) vẫn chạy bình thường, chỉ là ảnh không được làm nét thêm.')

In [ ]:
# Tự tắt nếu gfpgan không cài được ở cell trên; đổi thành False để bỏ qua thủ công.
# globals().get(...): cho phép bỏ qua HẲN mục 5 (không chạy cell patch/import nào) mà vẫn
# chạy tiếp được pipeline chính, thay vì NameError: GFPGAN_AVAILABLE.
USE_GFPGAN = globals().get('GFPGAN_AVAILABLE', False)

restorer = None
if USE_GFPGAN:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path='/content/models/GFPGANv1.4.pth',
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('GFPGAN sẵn sàng (chỉ chạy trên vùng crop quanh mặt đã swap).')
else:
    print('Bỏ qua GFPGAN, dùng ảnh swap gốc không phục hồi nét.')

## 6. Xử lý video: swap mặt từng frame (2 người, tracking theo danh tính)

**Vấn đề cần giải quyết:** model detect mặt mỗi frame **không đảm bảo thứ tự cố định** (frame này trả về [trái, phải], frame sau có thể trả về [phải, trái], đặc biệt khi 2 người quay đầu/che nhau lúc hôn). Nếu chỉ lấy theo index `[0]`, `[1]` thì mặt A/B sẽ bị **đảo lộn giữa các frame**, tạo hiệu ứng giật/lóe rất xấu.

**Cách xử lý ở đây — khớp theo *danh tính*, không chỉ theo vị trí:**

1. Ở frame đầu tiên detect được mặt, quy ước **A = mặt bên trái khung hình, B = mặt bên phải**, rồi lưu lại `normed_embedding` (vector nhận dạng ArcFace mà InsightFace **đã tính sẵn** ở bước detect — không tốn thêm chi phí) làm *danh tính tham chiếu* của từng người.
2. Mỗi frame sau, chi phí gán một khuôn mặt cho A/B = `(1 − cosine_similarity với embedding tham chiếu) + 0.25 × (khoảng cách vị trí đã chuẩn hoá)`. Embedding quyết định chính, vị trí chỉ phá hoà.
3. Mặt có `cosine_similarity < 0.20` bị **loại thẳng** — người lạ đi ngang hay người thứ 3 trong khung sẽ không bị swap nhầm (bản cũ luôn lấy mặt "gần nhất" nên không tránh được).
4. Việc gán được giải **tối ưu toàn cục** (duyệt hết các cách ghép, chỉ 2 người nên rất rẻ) thay vì greedy theo thứ tự A rồi B — greedy làm A luôn giành mặt trước, kể cả khi mặt đó rõ ràng là của B.

**Trường hợp 1 mặt bị che khuất (occlusion) hoàn toàn:** mặt hiện ra được gán cho **đúng người theo embedding**, người còn lại giữ nguyên frame gốc ở khung đó (không suy đoán mù).

**Người xuất hiện muộn:** nếu frame đầu chỉ detect được 1 mặt, người thứ hai vẫn được **khởi tạo muộn** ngay khi họ hiện ra ở frame sau.

**Ghi và ghép audio trong cùng một pass:** frame thô được ghi thẳng vào `ffmpeg` qua pipe, xuất ra H.264 kèm audio gốc. Bỏ được file trung gian `mp4v` (vốn làm mất chất lượng một lần trước khi re-encode lần hai) và bỏ luôn một lượt decode+encode toàn bộ video.

In [ ]:
import os, sys, itertools, subprocess
import numpy as np
import cv2
from tqdm import tqdm

# ---------------- Tham số tracking / chất lượng ----------------
POS_W           = 0.25   # trọng số vị trí trong hàm chi phí; embedding mới là yếu tố quyết định
MIN_SIM         = 0.20   # cosine similarity tối thiểu để chấp nhận "đúng người này"
MAX_DIST_RATIO  = 0.15   # bán kính chuẩn hoá khoảng cách, theo cạnh lớn của khung hình
EMB_EMA         = 0.05   # tốc độ cập nhật embedding tham chiếu
EMB_EMA_MIN_SIM = 0.50   # chỉ cập nhật khi khớp chắc chắn -> tránh trôi dần sang nhầm người
UNASSIGNED_COST = 1.5    # phạt khi để một người không được gán (lớn hơn mọi chi phí hợp lệ)
GFPGAN_PAD      = 0.4    # nới bbox bao nhiêu lần khi crop để restore

final_output = '/content/output_final.mp4'

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps <= 0:       # 0.0 hoặc NaN với một số container
    print('Không đọc được fps từ video, mặc định 25.')
    fps = 25.0

# CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho progress bar.
# Vòng lặp đọc tới khi hết frame thật sự, thay vì range(total_frames) (đếm thiếu = cụt đuôi video).
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

# Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có metadata
# rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg nhận rawvideo sai size.
ret, frame = cap.read()
assert ret, 'Không đọc được frame nào từ video.'
height, width = frame.shape[:2]
MAX_DIST = MAX_DIST_RATIO * max(width, height)

print(f'Video: {width}x{height} @ {fps:.2f}fps, ~{total_frames} frames')

# Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
ffmpeg_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}', '-r', f'{fps}', '-i', 'pipe:0',
    '-i', source_video_path,
    '-map', '0:v:0', '-map', '1:a:0?',        # '?' = không có audio thì bỏ qua, không lỗi
    '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
    '-pix_fmt', 'yuv420p',                    # để trình duyệt/IPython.display.Video phát được
    '-c:a', 'aac', '-shortest',
    final_output,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)


def face_center(face):
    x1, y1, x2, y2 = face.bbox
    return np.array([(x1 + x2) / 2.0, (y1 + y2) / 2.0])


def enhance_face_region(img, face, pad=GFPGAN_PAD):
    """Chỉ restore vùng quanh khuôn mặt vừa swap, KHÔNG chạy trên cả frame.

    Gọi restorer.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với app.get() vừa chạy), restore luôn cả những mặt
    trong nền không hề bị swap, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = restorer.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


# Danh tính tham chiếu (embedding) + vị trí gần nhất của từng người trong VIDEO.
ref_emb  = {'A': None, 'B': None}
last_pos = {'A': None, 'B': None}
source_face = {'A': source_face_a, 'B': source_face_b}


def sim_to(face, label):
    if ref_emb[label] is None:
        return None
    return float(np.dot(face.normed_embedding, ref_emb[label]))


def pos_cost(center, label):
    if last_pos[label] is None:
        return 1.0
    return min(float(np.linalg.norm(center - last_pos[label])) / MAX_DIST, 1.0)


def match_cost(face, center, label):
    """Chi phí gán `face` cho người `label`; None = không được phép gán."""
    s = sim_to(face, label)
    if s is None or s < MIN_SIM:
        return None          # khác người -> loại thẳng, không có chuyện "cứ gần nhất thì lấy"
    return (1.0 - s) + POS_W * pos_cost(center, label)


def solve_assignment(labels, faces, centers):
    """Gán tối ưu toàn cục (brute force — tối đa 2 người nên rất rẻ).

    Bản cũ duyệt candidates theo thứ tự cố định A rồi B, nên khi chỉ còn 1 mặt thì A luôn
    giành lấy, kể cả khi mặt đó rõ ràng gần/giống B hơn -> đảo danh tính lúc che khuất.
    """
    best_total, best_map = None, {}
    idx_choices = [None] + list(range(len(faces)))
    for combo in itertools.product(idx_choices, repeat=len(labels)):
        used = [c for c in combo if c is not None]
        if len(set(used)) != len(used):
            continue                                   # một mặt không thể là hai người
        total, mapping = 0.0, {}
        for label, idx in zip(labels, combo):
            if idx is None:
                total += UNASSIGNED_COST
                continue
            c = match_cost(faces[idx], centers[idx], label)
            if c is None:
                total = None
                break
            total += c
            mapping[label] = idx
        if total is None:
            continue
        if best_total is None or total < best_total:
            best_total, best_map = total, mapping
    return best_map


pbar = tqdm(total=total_frames, unit='frame')
frames_written = 0
rc = None
try:
    while frame is not None:
        target_faces = app.get(frame)
        result_frame = frame

        if target_faces:
            centers = [face_center(f) for f in target_faces]   # cache, tránh tính lại nhiều lần

            known = [l for l in ('A', 'B') if ref_emb[l] is not None]
            assigned = solve_assignment(known, target_faces, centers) if known else {}

            # Khởi tạo muộn: nếu frame đầu chỉ detect được 1 mặt, người thứ hai vẫn phải được
            # nhận diện khi họ hiện ra ở frame sau. Bản cũ chỉ khởi tạo đúng một lần ở frame đầu,
            # nên trong trường hợp đó B KHÔNG BAO GIỜ được swap trong cả video.
            unknown = [l for l in ('A', 'B') if ref_emb[l] is None]
            if unknown:
                taken = set(assigned.values())
                free = sorted((i for i in range(len(target_faces)) if i not in taken),
                              key=lambda i: centers[i][0])
                for label in unknown:                  # quy ước: A = bên trái, B = bên phải
                    if not free:
                        break
                    i = free.pop(0) if label == 'A' else free.pop(-1)
                    assigned[label] = i
                    ref_emb[label] = target_faces[i].normed_embedding.copy()
                    print(f'  [frame {frames_written}] khởi tạo danh tính {label}')

            for label, idx in assigned.items():
                face = target_faces[idx]
                last_pos[label] = centers[idx]
                # Cập nhật nhẹ embedding tham chiếu để bám theo thay đổi góc mặt/ánh sáng,
                # nhưng chỉ khi khớp chắc chắn -> tránh trôi dần sang nhầm người.
                s = sim_to(face, label)
                if s is not None and s >= EMB_EMA_MIN_SIM:
                    e = (1 - EMB_EMA) * ref_emb[label] + EMB_EMA * face.normed_embedding
                    ref_emb[label] = e / (np.linalg.norm(e) + 1e-8)
                result_frame = swapper.get(result_frame, face, source_face[label], paste_back=True)

            if restorer is not None:
                for idx in assigned.values():
                    result_frame = enhance_face_region(result_frame, target_faces[idx])

        proc.stdin.write(np.ascontiguousarray(result_frame).tobytes())
        frames_written += 1
        pbar.update(1)

        ret, frame = cap.read()
        if not ret:
            frame = None
finally:
    # Không release trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và file mp4 hỏng.
    pbar.close()
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    rc = proc.wait()

assert rc == 0, f'ffmpeg thất bại (exit code {rc}) — xem log lỗi ở trên.'
print(f'Đã swap {frames_written} frame và ghép audio xong -> {final_output}')

## 7. Kiểm tra kết quả

Audio đã được ghép ngay trong cell trên (ffmpeg nhận frame qua pipe và mux luôn audio gốc trong cùng một pass), nên ở đây chỉ cần xác nhận file xuất ra hợp lệ.

In [ ]:
import os

assert os.path.exists(final_output) and os.path.getsize(final_output) > 0, \
    'Không tạo được video output — chạy lại cell xử lý video ở mục 6.'
print(f'{final_output}  —  {os.path.getsize(final_output) / 1e6:.1f} MB\n')

# Xác nhận có stream video (và audio, nếu video gốc có audio)
!ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,r_frame_rate,duration \
  -of default=noprint_wrappers=1 {final_output}

## 8. Xem kết quả

In [ ]:
import os
from IPython.display import Video, display

size_mb = os.path.getsize(final_output) / 1e6
if size_mb > 50:
    # embed=True nhét toàn bộ file dưới dạng base64 vào output của notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài. Trường hợp đó thì tải về xem thay vì preview inline.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về máy.')
else:
    display(Video(final_output, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Hướng cải thiện tiếp theo

- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhòe nhẹ theo thời gian. Có thể cải thiện bằng cách thêm temporal smoothing (trung bình landmark giữa các frame liền kề) hoặc dùng model chuyên video như **SimSwap** với chế độ video.
- **Tracking 2 người**: pipeline dùng khớp theo embedding ArcFace (danh tính) + vị trí làm yếu tố phụ, giải bài toán gán tối ưu toàn cục mỗi frame. Cách này chịu được occlusion dài, đổi chỗ nhanh và cắt cảnh — những thứ mà tracking thuần theo vị trí hay gán nhầm. Nếu vẫn gặp nhầm lẫn, chỉnh `MIN_SIM` (tăng để khắt khe hơn) và `POS_W` ở cell mục 6.
- **Ai là A, ai là B**: quy ước lấy ở frame đầu tiên detect được mặt — A = mặt bên trái khung hình, B = mặt bên phải. Nếu bị ngược, chỉ cần đảo 2 ảnh upload ở bước 3.
- **Tốc độ**: xử lý frame-by-frame trên Colab free (T4) sẽ khá chậm với video dài. Nên test với video ngắn (5–10s) trước. GFPGAN chỉ chạy trên vùng crop quanh mặt đã swap (không phải cả frame) nên đã nhanh hơn đáng kể; tắt hẳn bằng `USE_GFPGAN = False` nếu vẫn quá chậm.
- **inswapper_128 model**: link tải có thể thay đổi do các vấn đề về chính sách/gỡ bỏ. Cell mục 2 đã tự kiểm tra dung lượng file và báo lỗi ngay nếu tải hỏng, thay vì để tới lúc load model mới nổ.
- **Chất lượng ảnh mặt nguồn**: ảnh càng rõ, chính diện, ánh sáng đều thì kết quả swap càng tự nhiên.